# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第2章：音声データの記述統計と分布の可視化


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本書の巻末に載っているGitHubリポジトリのリンクから、「Open in Colab」のバッジで開いている。

「このノートブックはGoogleが作成したものではありません」という警告は、外部（GitHub）から
読み込むノートブック全般に出る定型の警告である。内容を確認のうえ実行してよい。
書き込んだ内容を残したいときは、「ファイル」→「ドライブにコピーを保存」で自分のGoogle Driveに
保存する。保存したコピーは自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照論文: Peterson & Barney (1952), *JASA*／Winter (2019) *Statistics for Linguists*, 第3-4章。

■ データの読み込み：

第1章で作った共通データセットを読み込む。

In [ ]:
# ---- 共通データセットの読み込み ----
# 第1章と全く同じ手順・同じ乱数seedでVOTデータを作り直す。
# 「本ごとに違うデータ」を避けるためだが、外部URLから読み込むのではなく
# その場で再生成する設計にしてある——理由は、配布用ノートブックが
# 非公開のGitHubリポジトリのURLに依存すると、読者の環境からは
# アクセスできず404になってしまうため（実際にこの問題が起きて修正した）。
# コードを自分の環境で動かす分には、この方式のほうが「ネットワーク不要で
# 必ず同じ結果になる」という利点もある。
if (!requireNamespace("tidyverse", quietly = TRUE)) install.packages("tidyverse", repos = "https://cloud.r-project.org")
library(tidyverse)
# ---- 図の日本語文字化け対策・文字サイズの既定値 ----
# Colabの既定フォントには日本語グリフが含まれないため、図のタイトルや軸ラベルが
# 文字化け（豆腐記号）して表示される。Google Fontsの日本語フォントを取得し、
# "notosans"という名前で登録して使う（要ネット接続。Colabでは常に使える）。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")  # "sans"などの予約名は上書きできないため専用名で登録
showtext_auto()
showtext_opts(dpi = 100)                     # Colabのインライン画像解像度(100dpi)に合わせる
par(family = "notosans")                     # base Rの作図（mclust・itsadugなど）にも適用する
theme_set(theme_minimal(base_size = 14, base_family = "notosans"))  # ggplotの既定テーマ


set.seed(42)  # 乱数のseedを固定する。同じ数字なら誰が実行しても同じ結果になる
n_speaker <- 10                    # 話者数
n_word    <- 20                    # 単語数（各話者が全単語を1回ずつ発話する）
n_obs     <- n_speaker * n_word    # = 200行

# 何行目がどの話者・どの単語かを表す添字。あとで話者ごと・単語ごとの
# 値を引くのに使う（speaker_intercept[speaker_idx] のように書ける）。
speaker_idx <- rep(1:n_speaker, each  = n_word)
word_idx    <- rep(1:n_word,    times = n_speaker)

# 1行ごとにvoiced/voicelessを交互に並べる。話者1人あたり20行なので、
# どの話者を見ても必ずvoiced10回・voiceless10回が含まれる
# ——これは第5章で「話者ごとにconditionの効果を推定する」ために必須の設計。
condition_vec <- rep(c("voiced", "voiceless"), times = n_obs / 2)

# ---- 話者差・単語差を「先に」作っておく ----
# 実際の音声データでは、同じ条件でも話者によってVOTの平均が違い（声道長・
# 発話速度・方言）、単語によっても違う（後続母音・語長・頻度）。
# 擬似データでもこの構造を明示的に作っておかないと、第5章以降で使う
# 混合効果モデルが「推定すべきばらつきが最初から存在しない」データを
# 相手にすることになり、分散推定値がすべて0（singular fit）になってしまう。
speaker_intercept <- rnorm(n_speaker, mean = 0, sd = 6)
# 話者ごとのVOTの底上げ量。どの条件にも一律にかかる（＝ランダム切片）。
speaker_slope <- rnorm(n_speaker, mean = 0, sd = 10)
# 話者ごとの「有声と無声をどれだけ大きく作り分けるか」の差（＝ランダム傾き）。
# 話者間のばらつきは無声（VOTが長い側）で特に大きいことが知られているので、
# voiceless条件にだけ加える。
word_intercept <- rnorm(n_word, mean = 0, sd = 3)
# 単語ごとの癖（＝項目のランダム切片）。

vot_data <- tibble(
  speaker   = paste0("S", sprintf("%02d", speaker_idx)),
  word      = paste0("word_", word_idx),
  gender    = rep(rep(c("M", "F"), each = n_speaker / 2), each = n_word),
  condition = condition_vec,
  # VOTは「固定効果 + 話者差 + 単語差 + 残差」を足し合わせて作る。
  # これは第5章で当てはめる混合効果モデルの式そのものであり、
  # モデルが推定した分散をこの生成時の値と見比べられるようにしてある。
  VOT = 20 +                                              # 有声条件の基準値
    if_else(condition == "voiceless", 55, 0) +            # 無声条件の上乗せ（→約75ms）
    speaker_intercept[speaker_idx] +                      # 話者のランダム切片
    if_else(condition == "voiceless",                     # 話者のランダム傾き
            speaker_slope[speaker_idx], 0) +
    word_intercept[word_idx] +                            # 単語のランダム切片
    # 残差。無声条件のほうがばらつきが大きい（実際のVOTデータもそうなっている）。
    # 第3章のモデル診断で「等分散性が崩れている」例として使う。
    rnorm(n_obs, mean = 0, sd = if_else(condition == "voiced", 4, 8)),
  F0 = rnorm(n_obs, mean = 130, sd = 15)  # F0はここでは条件と無関係に生成
)

vot_data <- vot_data %>%
  mutate(
    # 文字列のままだとRが「順序のない値」として扱うため、
    # 話者・単語・性別・条件をすべてfactor（因子）型に変換しておく。
    # 特にconditionはlevels=でvoiced/voicelessの順序を明示し、
    # 以降のモデルの切片が「voiced群の平均」になるよう固定する。
    speaker   = factor(speaker),
    word      = factor(word),
    gender    = factor(gender),
    condition = factor(condition, levels = c("voiced", "voiceless"))
  )

glimpse(vot_data)  # 型と最初の数行を確認。数値であるべき列が<chr>のままなら要注意

# --- 実行結果 ---
# Rows: 200
# Columns: 6
# $ speaker   <fct> S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, ...
# $ word      <fct> word_1, word_2, word_3, word_4, word_5, word_6, word_7, ...
# $ gender    <fct> M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, ...
# $ condition <fct> voiced, voiceless, voiced, voiceless, voiced, voiceless, ...
# $ VOT       <dbl> 28.129829, 88.042064, 30.742652, 94.104833, 28.438207, ...
# $ F0        <dbl> 119.0617, 144.9710, 148.8772, 148.7330, 109.2904, ...
#
# 【出力の読み方】
# ・Rows: 200 / Columns: 6 が行数と列数。10話者×20語=200行になっているかを
#   まず確認する。ここが合っていなければ、以降の分析はすべて無意味になる。
# ・各行の <fct> <dbl> が**型**である。<fct>はfactor（水準を持つカテゴリ）、
#   <dbl>は小数を含む数値。speaker・word・gender・conditionがすべて<fct>、
#   VOTとF0が<dbl>になっていれば正しい。
#   **VOTが<chr>（文字列）になっていたら、そこで止まって読み込みを見直す。**
#   文字列のままでは平均もモデルも計算できず、しかもエラーではなく
#   「変な結果」として静かに進んでしまうことがある。
# ・conditionの並びがvoiced, voiceless, voiced, voiceless...と1行おきに
#   交互になっている。これは意図した設計どおり（どの話者も両条件を10回ずつ持つ）。
# ・speakerが先頭でS01ばかり並ぶのは、話者ごとに20行がまとまっているため。
#   行の並び順自体は分析結果に影響しない（第7章のAR(1)を使うときだけ例外）。


■ 記述統計：分布の形を見る：

**実際の研究例**：Peterson & Barney (1952) の母音フォルマント研究は、76名の話者（男性33名・女性28名・子ども15名）が発音した10母音のF1・F2を測定し、二次元平面上にプロットした。各母音は明確なクラスタを作りながらも、互いに重なり合う楕円状の分布を示した。もし平均F1・平均F2だけを報告していたら、母音空間の「重なり」——聞き手が誤って聞き取りうる領域——という、その後の音声知覚研究の出発点になった知見は見えなかった。**平均値は分布の要約に過ぎず、分布の形そのものが情報を持つ。**

Winter (2019, 第3-4章) の方針にならい、平均・SDだけでなく歪度・分位点も確認する。

In [ ]:
vot_data %>%
  group_by(condition) %>%
  summarise(
    mean = mean(VOT), sd = sd(VOT),
    q25 = quantile(VOT, .25), median = median(VOT), q75 = quantile(VOT, .75),
    # 歪度（skewness）: 分布が左右非対称にどれだけ傾いているかの指標。
    # 正なら右に裾を引く（大きい値の外れ値が多い）、負なら左に裾を引く。
    # VOTのような「0msに近づけない」時間指標は、正の歪度を持ちやすい。
    skewness = mean((VOT - mean(VOT))^3) / sd(VOT)^3,
    .groups = "drop"
  )

# --- 実行結果 ---
# # A tibble: 2 x 7
#   condition  mean    sd   q25 median   q75 skewness
#   <fct>     <dbl> <dbl> <dbl>  <dbl> <dbl>    <dbl>
# 1 voiced     22.9  6.35  18.7   22.3  27.7   0.0734
# 2 voiceless  75.4 17.3   64.8   77.0  88.8  -0.399
#
# 【出力の読み方】
# ・q25 / median / q75 は下から25%・50%・75%の位置にある値。
#   q25とq75の間に観測のちょうど半分が入る（この幅を四分位範囲IQRと呼ぶ）。
#   voicedはIQR=27.7-18.7=9.0ms、voicelessは88.8-64.8=24.0msで、
#   **無声条件のほうが2倍以上広い**。sdの6.35対17.3と同じことを言っている。
# ・meanとmedianを見比べる。voicedは22.9対22.3、voicelessは75.4対77.0で、
#   どちらも差は1.6ms以内。平均と中央値が近いのは分布が概ね対称という合図で、
#   このデータなら平均を代表値として報告してよい、と判断できる。
# ・skewness（歪度）は0が完全な左右対称。正なら右（大きい値の側）に裾を引き、
#   負なら左に裾を引く。voicedの0.0734はほぼ0で対称。voicelessの-0.399は
#   わずかに左に裾を引いている（＝短めの外れ値が少しある）。
#   目安として絶対値が0.5未満なら「ほぼ対称」、1を超えたら歪みを疑う。
# ・上のコメントに「時間指標は正の歪度を持ちやすい」と書いたが、ここでは
#   voicelessが負に出ている。**擬似データを正規分布から作っているので、
#   実データで期待される右の裾が再現されていない**ためである。
#   自分の実測データでは正の歪度になることのほうが多い——そのときこそ、
#   平均ではなく中央値を報告するか、対数変換を検討する場面になる。
# ・歪度の値そのものより、**meanとmedianの差・sdとIQRの関係と合わせて
#   「この分布に平均を使ってよいか」を判断する**ために見る指標である。


■ 図表の選び方（統計解析の前に）：

モデルを当てる前に必ず分布を見る、というのがこの章の一貫した主張である。**ただし「何でもいいから図を描け」では役に立たない。見たい問い・変数の型に応じて、描くべき図の種類は決まっている。** 表にまとめておく。

| 見たいこと | 変数の型 | 向いている図 |
|---|---|---|
| 1つの連続量の分布の形 | 連続量1つ | ヒストグラム／密度プロット |
| カテゴリ間で分布を比べる | 連続量×カテゴリ | 箱ひげ図／バイオリンプロット
（＋個々の点を`geom_jitter()`で重ねる） |
| 2つの連続量の関係 | 連続量×連続量 | 散布図（＋回帰直線・楕円） |
| カテゴリの度数・構成比 | カテゴリ1つ | 棒グラフ（積み上げなら比率も） |
| 2つのカテゴリ変数の関係 | カテゴリ×カテゴリ | グループ化棒グラフ |
| 順序や時間に沿った変化 | 連続量×順序・時間 | 折れ線グラフ |

**円グラフは、この表のどこにも挙げていない。** カテゴリの構成比を見せたい場面でも避ける。人間は角度や面積の大小を正確に読み取るのが苦手で、長さ（棒の高さ）の比較のほうがずっと精度よく行えることが知覚実験の研究で繰り返し確認されている（Cleveland & McGill 1984）。要素が3つを超えると角度の差はほぼ読み取れなくなる。同じ情報は棒グラフのほうが正確に伝わる。

以下、上の表の主な型を順に、`vot_data`を使って実演する。

■ ヒストグラムと密度プロット：

In [ ]:
ggplot(vot_data, aes(x = VOT, fill = condition)) +
  # after_stat(density) で「度数」ではなく「密度」を縦軸にし、
  # ビン幅が違ってもヒストグラムと密度曲線の面積が揃うようにする
  geom_histogram(aes(y = after_stat(density)), alpha = 0.5, position = "identity", bins = 20) +
  geom_density(alpha = 0.3) +
  facet_wrap(~condition, scales = "free_x") +  # 条件ごとに別パネルに分ける
  theme_minimal(base_size = 14, base_family = "notosans")

# --- 実行結果 ---
# 図が1枚（左右2パネル）描画される。テキスト出力は無い。
#
# 【図の読み方】
# ・左パネルがvoiced、右パネルがvoiceless。棒が度数分布（ヒストグラム）、
#   上に重なる曲線がカーネル密度推定である。**山が1つなら単峰性**で、
#   平均を代表値として使ってよい状況だと判断できる。
# ・縦軸は「度数」ではなく「密度」になっている。after_stat(density)を指定すると
#   棒の**面積の合計が1**になるよう縦軸が調整され、ビン幅や件数が違っても
#   ヒストグラムと密度曲線を同じ縮尺で重ねられる。
#   したがって縦軸の数値そのものを「何件」と読んではいけない。
# ・scales = "free_x" により左右のパネルで横軸の範囲が別々に決まる。
#   分布の**形**を見るには便利だが、**2条件の位置の違いは読み取れなくなる**。
#   位置を比べたいときはこの指定を外すか、前のセルのバイオリンプロットを使う。
# ・右パネル（voiceless）のほうが横に広がって見えるのは、sdが17.3と大きいため。
#   ただし横軸の範囲が違うので、幅の見た目だけで比較しないこと。


■ 母音空間のような2次元プロット（F0 × VOT）：

音声学では2つの音響指標の関係（母音空間ならF1×F2）を散布図で見る場面が多い。

In [ ]:
ggplot(vot_data, aes(x = VOT, y = F0, color = condition)) +
  geom_point(alpha = 0.6) +
  # stat_ellipse(): 各groupのデータが従う2変量正規分布を仮定し、
  # 指定した確率（level）を含む楕円を描く。Peterson & Barneyの母音空間図と同じ発想。
  stat_ellipse(level = 0.68) +
  theme_minimal(base_size = 14, base_family = "notosans") +
  labs(title = "VOTとF0の関係（68%信頼楕円）")

# --- 実行結果 ---
# 散布図が1枚描画される。テキスト出力は無い。
#
# 【図の読み方】
# ・横軸VOT、縦軸F0で、点1つが1観測（1話者の1語）。色が条件を表す。
# ・2色の点の塊は**横方向にははっきり離れている**（VOTが違う）が、
#   **縦方向には同じ高さに重なっている**（F0は違わない）。
#   本書のデータはF0を条件と無関係に生成しているので、これが正しい姿である。
#   もし縦方向にもずれて見えたら、データかコードを疑う。
# ・楕円は「2変量正規分布を仮定したとき、観測の68%が入る範囲」を描いたもの。
#   level = 0.68 は1標準偏差におおよそ対応する。**データを囲む枠ではない**ので、
#   楕円の外に3割ほどの点がはみ出すのが正常である。
# ・楕円の傾きが2変数の相関を表す。ここでは2つともほぼ水平で、
#   VOTとF0に相関が無いことを示している。傾いていれば、片方が大きいほど
#   もう片方も大きい（または小さい）という関係があることになる。
# ・Peterson & Barney (1952) の母音空間図（横軸F1・縦軸F2に母音ごとの楕円を描く）
#   と同じ描き方である。軸を第1・第2フォルマントに変えれば、そのまま
#   母音のプロットとして使える。


■ 自分の1点を全体の分布の中に位置づける：

音声学の教科書や発表資料でよく見る図に、**「全体の分布の中に、自分（または特定の話者）の測定値を1点だけ強調して示す」**というものがある。母音空間図に自分の発音した /i/ を赤い点で重ねて「自分の発音は平均的な範囲に入っているか」を示す、といった使い方である。ここでは`vot_data`の中の1トークンを「自分の観測値」に見立てて同じことをする。

In [ ]:
# 全観測点を薄い色で描き、その上に強調したい1点だけ目立つ色・大きさで重ねる。
# ★ポイントは、強調用の点を「別のgeom_point()」として2回目に描くこと。
#   1回のgeom_point()の中でaes(size=...)を条件分岐させるより、
#   レイヤーを分けたほうがコードが素直で、凡例も作りやすい。
my_token <- vot_data %>% filter(speaker == "S01", word == "word_1")

ggplot(vot_data, aes(x = VOT, y = F0)) +
  geom_point(aes(color = condition), alpha = 0.35, size = 1.8) +   # 全体（背景）
  geom_point(data = my_token, color = "red", size = 4) +           # 強調する1点
  geom_text(data = my_token, aes(label = "S01 / word_1"),
            hjust = -0.15, vjust = 0.3, color = "red", size = 3.5) +
  theme_minimal(base_size = 14, base_family = "notosans") +
  labs(title = "1トークンを全体の分布の中で確認する",
       x = "VOT (ms)", y = "F0 (Hz)")

cat("強調した点: VOT =", round(my_token$VOT, 1), "ms, F0 =", round(my_token$F0, 1), "Hz\n")
cat("全体平均　: VOT =", round(mean(vot_data$VOT), 1), "ms, F0 =", round(mean(vot_data$F0), 1), "Hz\n")

# --- 実行結果 ---
# 強調した点: VOT = 28.1 ms, F0 = 119.1 Hz
# 全体平均　: VOT = 49.1 ms, F0 = 129.8 Hz
#
# 【図の読み方】
# ・背景の全観測点を先に薄く描き、強調したい1点を**後から上に重ねて描く**のが
#   コツである。ggplot2はレイヤーを追加した順に上へ重ねるので、後に描いた
#   `geom_point(data = my_token, ...)` が必ず前面に来る。
# ・強調点のS01/word_1はVOT=28.1msと、有声条件の中でも短めの値である。
#   全体平均（49.1ms）は有声・無声を混ぜた値なので、「強調点がどちらの
#   条件のかたまりに属しているか」を見てから比べないと誤読する。
# ・**`data = my_token` のように、geomごとに違うデータフレームを渡せる**のが
#   ggplot2の重要な性質である。全体は`vot_data`、強調点は`my_token`という
#   1行だけの別データフレームを渡している。自分のデータを既存の分布に
#   重ねたいときは、常にこの形になる。
# ・母音空間図でも同じ発想が使える。全話者の母音分布を薄く描いた上に、
#   自分（または注目する1話者）の発音を`geom_point(data = ..., color = "red")`
#   で重ねれば、自分の発音がどの母音カテゴリの範囲に入っているか、
#   カテゴリ間のどのあたりに位置するかが一目でわかる。


■ 基本的なグラフの型と、音声学での使い分け：

ggplot2にはさまざまな`geom_*()`があるが、音声学の記述統計で実際によく使うのは次の3種類にほぼ絞られる。

- **棒グラフ（bar chart）**: カテゴリごとの集計値（平均・件数など）を比べる
- **折れ線グラフ（line chart）**: 時間や連続量の推移を見る（第7章のF0軌跡など）
- **ヒストグラム／密度プロット**: 1つの連続変数の分布の形を見る（本章で既出）

**円グラフ（pie chart）は本書では意図的に使わない。** 円グラフは扇形の角度・面積を目でくらべる必要があり、棒グラフに比べて「どちらが大きいか」を正確に読み取りにくいことが可視化研究で繰り返し指摘されている。同じ情報は棒グラフで表現でき、しかもより正確に伝わるため、選択肢がある場面では常に棒グラフを使う。

In [ ]:
# 棒グラフ：条件ごとの平均VOTを比べる
vot_data %>%
  group_by(condition) %>%
  summarise(mean_VOT = mean(VOT), .groups = "drop") %>%
  ggplot(aes(x = condition, y = mean_VOT, fill = condition)) +
  geom_col() +  # geom_col()は「すでに集計済みの値」を棒の高さにする
               # （集計前の生データの件数を数えたいだけならgeom_bar()を使う）
  theme_minimal(base_size = 14, base_family = "notosans") +
  labs(title = "条件ごとの平均VOT", y = "平均VOT (ms)")

# --- 実行結果 ---
# 棒グラフが1枚描画される。テキスト出力は無い。
#
# 【図の読み方】
# ・棒の高さはそれぞれ約22.9msと約75.4ms（前のセルの表のmeanと一致する）。
# ・**この図は本書で最も情報量の少ない図である。** 棒2本が示すのは平均だけで、
#   ばらつきも、分布の形も、話者ごとの違いも、観測が何件あるかも見えない。
#   同じデータから作ったバイオリンプロット（第1章）と見比べると、
#   棒グラフが何を捨てているかがわかる。
# ・論文の図として棒グラフだけを出すと、査読で「分布を示せ」と言われる。
#   最低でもエラーバー（標準誤差や信頼区間）を足し、可能なら
#   geom_jitter()で個々の点を重ねる。第1章・第2章の演習で試したとおりである。
# ・geom_col()とgeom_bar()の違いに注意する。ここでは先にsummarise()で平均を
#   計算済みなので、その値をそのまま高さにするgeom_col()を使う。
#   geom_bar()は既定で「行数を数える」ので、集計済みの値に使うと意図が狂う。


■ 2つのカテゴリ変数の関係を見る：グループ化棒グラフ：

「条件（voiced/voiceless）ごとの度数」のような**1つの**カテゴリ変数なら`geom_bar()`だけでよいが、**2つの**カテゴリ変数（例: 条件と性別）の組み合わせを見たい場面では、`fill`で色分けした上で`position = "dodge"`（棒を横に並べる）を指定する。

In [ ]:
# geom_bar(): 集計前の生データから「行数（度数）」を自動で数える。
# ここでは条件×性別の組み合わせごとの件数を、棒を横に並べて比較する。
ggplot(vot_data, aes(x = condition, fill = gender)) +
  geom_bar(position = "dodge") +   # "dodge": 同じx位置で棒を横に並べる（積み上げない）
  theme_minimal(base_size = 14, base_family = "notosans") +
  labs(title = "条件×性別の観測数", y = "件数")

# --- 実行結果 ---
# グループ化棒グラフが1枚描画される。テキスト出力は無い。
#
# 【図の読み方】
# ・4本の棒（voiced×M, voiced×F, voiceless×M, voiceless×F）はすべて高さ50で
#   揃う。本書のデータは話者5名ずつ・各話者が両条件を10回ずつ発話する設計
#   なので、これは「設計どおり均等に割り付けられている」ことの確認になる。
#   実データでこの図を描いて棒の高さが不揃いなら、収集や除外の過程で
#   偏りが生じていないかをまず疑う。
# ・`position = "dodge"`を`"stack"`に変えると、同じx位置で棒が積み上がり、
#   合計の高さと内訳が同時に見える形になる。**どちらが良いかは目的次第**
#   で、「グループ間の絶対数を比べたい」ならdodge、「全体に占める割合の
#   感覚をつかみたい」ならstackが向く。
# ・性別が3水準以上（本書のPeterson & Barneyデータのように男性・女性・
#   子どもの3群）に増えても、同じコードでそのまま棒が3本に増えるだけで
#   済む。これがgeom_bar()／geom_col()を使う理由の一つである——手で
#   数えて表を作る必要がない。


In [ ]:
# 折れ線グラフ：話者ごとの「1トークン目から20トークン目までのVOTの並び」を見る例。
# 音声学ではこの形を、時間軸に沿ったF0やフォルマントの推移を見るのにも使う
# （第7章でより本格的に扱う）。
vot_data %>%
  filter(speaker == "S01") %>%
  mutate(token_index = row_number()) %>%
  ggplot(aes(x = token_index, y = VOT, color = condition, group = 1)) +
  geom_line(color = "grey60") +   # 折れ線本体
  geom_point(size = 2) +          # 各点を強調
  theme_minimal(base_size = 14, base_family = "notosans") +
  labs(title = "話者S01のVOTの推移（発話順）", x = "発話順", y = "VOT (ms)")

# --- 実行結果 ---
# 折れ線グラフが1枚描画される。テキスト出力は無い。
#
# 【図の読み方】
# ・横軸は話者S01の発話順（1〜20）、縦軸はそのときのVOT。
# ・線が**1つおきに大きく上下するジグザグ**になる。これはS01の発話が
#   時間とともに変動しているのではなく、第1章でvoiced/voicelessを
#   1行おきに交互に並べたためである。低い点がvoiced、高い点がvoiceless。
#   点の色を見れば条件が交互であることが確認できる。
# ・**したがってこの図から「発話を重ねるほどVOTが伸びた」といった時間変化を
#   読み取ってはいけない。** 横軸に並べた順序に意味がない場合、折れ線は
#   存在しない傾向を見せてしまう。折れ線グラフは、横軸が本当に順序または
#   連続量であるときにだけ使う。
# ・group = 1 を指定しているのは、色（condition）ごとに線が分断されるのを
#   防いで1本につなぐため。ggplot2は既定で色分けごとに別の線を引く。
# ・横軸が「時間」として意味を持つ本物の例——F0やフォルマントの軌跡——は
#   第7章で扱う。そこでは折れ線が正しい選択になる。


■ 【発展・参考】波形・スペクトログラム・スペクトルの表示：

本書のVOTやF0は「すでに測定し終えた数値」を扱っているが、実際の研究ではその手前に音響信号そのものを目で確認する作業がある。用語を整理しておく。

- **波形（waveform）**: 縦軸=振幅、横軸=時間。音圧の変化をそのまま描いたもの。破裂音の閉鎖区間や気音の開始点を目視で確認するときに見る
- **スペクトログラム（spectrogram）**: 縦軸=周波数、横軸=時間、色や濃淡=強度。フォルマント（母音の共鳴周波数）の推移や、摩擦音のエネルギー分布はここで読む
- **スペクトル（spectrum）**: 縦軸=強度、横軸=周波数。ある一瞬（あるいは区間の平均）における周波数成分の分布。スペクトル重心（spectral moments）などの指標はここから計算する

これらはRの標準機能ではなく、`tuneR`・`seewave`のような音声信号処理用のパッケージが必要になる。以下は参考コード（**本書のCSVデータには対応する音声ファイルが無いため、このセルはこのノートブックでは実行できない**。自分のwavファイルがある場合にのみ、パスを差し替えて試すこと）。

```r
install.packages(c("tuneR", "seewave"), repos = "https://cloud.r-project.org")
library(tuneR); library(seewave)

snd <- readWave("自分の録音.wav")  # wavファイルを読み込む

plot(snd)                    # 波形
spectro(snd)                 # スペクトログラム
meanspec(snd)                # 平均スペクトル
```

実際の音響測定（VOT・フォルマント・F0の自動抽出）を大規模に行いたい場合は、PraatをRから操作する`PraatR`や、Pythonの`parselmouth`（Praatのラッパー）を使う研究者が多い。本書はその**後段**——測定し終えた数値をどう統計的に扱うか——に主眼を置いている。

■ 実データで母音空間を描く（Peterson & Barney 1952）：

ここまでは擬似データを使ってきたが、本節だけは**本物の実測データ**を使う。`phonTools`パッケージには、Peterson & Barney (1952) が測定した**76名の話者（男性33名・女性28名・子ども15名）が10母音を2回ずつ発音した1,520トークン**のフォルマント値がそのまま収録されている。音響音声学で最も有名なデータセットであり、母音空間図の原型になったものである。

母音記号はX-SAMPA風の表記になっている（`i`=/i/, `I`=/ɪ/, `E`=/ɛ/, `{`=/æ/, `A`=/ɑ/, `O`=/ɔ/, `U`=/ʊ/, `u`=/u/, `V`=/ʌ/, `3'`=/ɝ/）。

In [ ]:
# phonToolsパッケージに Peterson & Barney (1952) の実測データが入っている
install.packages("phonTools", repos = "https://cloud.r-project.org")
library(phonTools)

data(pb52)   # 1520行 x 9列
str(pb52)

# type列は話者の種別（m=男性, w=女性, c=子ども）。日本語ラベルを付けておく。
pb <- pb52 %>%
  mutate(talker_type = recode(as.character(type), m = "男性", w = "女性", c = "子ども"))

# 母音ごとの平均フォルマント。F1が低いほど「狭い（高い）」母音、
# F2が高いほど「前寄り」の母音になる。
pb %>%
  group_by(vowel) %>%
  summarise(n = n(), F1 = round(mean(f1)), F2 = round(mean(f2)), .groups = "drop") %>%
  arrange(F1)


■ 母音空間図の描き方（軸を反転させる理由）：

In [ ]:
# 母音空間図の約束事: **横軸にF2、縦軸にF1を取り、どちらも軸を反転させる。**
# こうすると、図の左右が舌の前後（左=前舌）、上下が舌の高さ（上=狭い母音）に
# 対応し、伝統的な母音四辺形と同じ配置になる。
# scale_x_reverse() / scale_y_reverse() を忘れると、上下左右が逆の図になり、
# 音声学の読者には「読めない図」になってしまう。
ggplot(pb, aes(x = f2, y = f1, color = vowel)) +
  geom_point(alpha = 0.25, size = 0.8) +
  stat_ellipse(level = 0.68, linewidth = 0.7) +  # 各母音の68%楕円
  scale_x_reverse() +
  scale_y_reverse() +
  labs(title = "Peterson & Barney (1952) の母音空間",
       x = "F2 (Hz)  ←前舌　　後舌→", y = "F1 (Hz)  ←狭い　　広い→") +
  theme_minimal(base_size = 14, base_family = "notosans")

# --- 実行結果 ---
# 母音空間図が1枚描画される。テキスト出力は無い。
#
# 【図の読み方】
# ・点1つが1トークン（ある話者のある母音の1回の発音）。色が母音カテゴリ。
#   左上に /i/、右下に /A/ が来て、全体が四辺形に広がる。
# ・**楕円が大きく重なっていることが、この図の一番重要な情報である。**
#   /I/ と /E/、/A/ と /O/ などは、F1とF2だけでは分離できていない。
#   Peterson & Barney自身が報告したとおり、聴き手が正しく同定できる母音でも、
#   フォルマント2つだけでは重なる。**「母音は音響的にきれいに分かれている」
#   という素朴な期待は、実データでは成り立たない。**
# ・重なりの主な原因は、76名の話者の声道長がばらばらなことにある。
#   次のセルでその中身を分解する。


■ ★よくある誤り：正規化せずに話者をまたいで比べる：

母音研究で最も多い誤りの1つが、**声道長の違う話者のフォルマント値を生のHzのまま比較してしまう**ことである。同じ音素でも、子どもは声道が短くフォルマントが全体的に高く出る。これを「発音が違う」と読むと、**実際には解剖学的な差でしかないものを言語的な差として報告する**ことになる。以下は、その誤りと対処を同じデータで並べて確かめるコードである。

In [ ]:
# ---- 誤った分析: 生のHzのまま話者タイプを比べる ----
# 「子どもの /i/ は男性の /i/ より前寄りに調音されている」と言いたくなる場面。
raw_i <- pb %>%
  filter(vowel == "i") %>%
  group_by(talker_type) %>%
  summarise(F1 = round(mean(f1)), F2 = round(mean(f2)), .groups = "drop")
print(raw_i)
cat("子ども vs 男性 のF2差:",
    raw_i$F2[raw_i$talker_type == "子ども"] - raw_i$F2[raw_i$talker_type == "男性"], "Hz\n")

# ---- 正しい分析: 話者ごとに正規化してから比べる ----
# Lobanov正規化: 話者ごとに、その人の全母音の平均とSDを使ってzスコア化する。
# 「その話者の母音空間の中で、この母音はどのあたりか」に置き換える操作である。
pb_norm <- pb %>%
  group_by(speaker) %>%
  mutate(f1z = as.numeric(scale(f1)),
         f2z = as.numeric(scale(f2))) %>%
  ungroup()

norm_i <- pb_norm %>%
  filter(vowel == "i") %>%
  group_by(talker_type) %>%
  summarise(F1z = round(mean(f1z), 3), F2z = round(mean(f2z), 3), .groups = "drop")
print(norm_i)
cat("子ども vs 男性 のF2z差:",
    round(norm_i$F2z[norm_i$talker_type == "子ども"] - norm_i$F2z[norm_i$talker_type == "男性"], 3),
    "SD\n")

# ---- どちらが「母音の違い」を捉えられているかを数字で比べる ----
cat("\n話者タイプがF2のばらつきを説明する割合(R^2):\n")
cat("  生のHz    :", round(summary(lm(f2  ~ talker_type, data = pb_norm))$r.squared, 4), "\n")
cat("  正規化後  :", round(summary(lm(f2z ~ talker_type, data = pb_norm))$r.squared, 4), "\n")
cat("母音がF2のばらつきを説明する割合(R^2):\n")
cat("  生のHz    :", round(summary(lm(f2  ~ vowel, data = pb_norm))$r.squared, 4), "\n")
cat("  正規化後  :", round(summary(lm(f2z ~ vowel, data = pb_norm))$r.squared, 4), "\n")

# --- 実行結果 ---
#   talker_type    F1    F2
# 1 女性          310  2783
# 2 子ども        360  3178
# 3 男性          267  2294
# 子ども vs 男性 のF2差: 884 Hz
#
#   talker_type   F1z   F2z
# 1 女性        -1.34  1.64
# 2 子ども      -1.33  1.71
# 3 男性        -1.54  1.73
# 子ども vs 男性 のF2z差: -0.023 SD
#
# 話者タイプがF2のばらつきを説明する割合(R^2):
#   生のHz    : 0.0931
#   正規化後  : 0
# 母音がF2のばらつきを説明する割合(R^2):
#   生のHz    : 0.834
#   正規化後  : 0.9673
#
# 【出力の読み方】
# ・生のHzでは、子どもの /i/ のF2は男性より **884Hzも高い**。これだけ見れば
#   「子どもは /i/ をはるかに前寄りに調音する」という結論になりそうである。
# ・ところが正規化すると、同じ差が **-0.023 SD**、つまり実質ゼロになる。
#   しかも符号まで逆転している。**884Hzの差は、子どもの声道が短いという
#   解剖学的事実がそのまま出ていただけ**で、調音方法の違いではなかった。
# ・R²の比較がこれを裏づける。生のHzでは話者タイプがF2のばらつきの9.3%を
#   説明してしまうが、正規化後は0%になる（話者内でzスコア化するので当然だが、
#   「消すべきものを消せている」ことの確認になる）。
# ・同時に、**母音カテゴリの説明力は83.4%から96.7%へ上がっている。**
#   正規化は情報を捨てる操作ではなく、**話者差というノイズを取り除いて
#   母音の違いを見えやすくする操作**である。前のセルで楕円が重なって
#   見えたのは、この話者差が混ざっていたためだった。
# ・★ただし正規化は万能ではない。Lobanov正規化は「どの話者も同じ母音の
#   セットを同じ回数発音している」ことを前提にしている。話者ごとに
#   取れた母音が違うデータに当てると、**平均とSDの基準がずれて
#   別の歪みを持ち込む**。方言比較のように母音体系そのものが違う場合も同様で、
#   正規化してよいかどうかは研究の問いから判断する必要がある。
# ・★もう1つの誤り: 「話者差は正規化で消したから、あとは普通の回帰でよい」
#   と考えること。正規化しても**同じ話者から複数のトークンを取っている**
#   事実は変わらないので、第5章の混合効果モデルは依然として必要である。


■ 母音四辺形（頂点を平均値で結ぶ、教科書でよく見る形）：

音声学の教科書でよく見るもう1つの母音空間図は、個々のトークンの散布ではなく、**母音カテゴリごとの平均フォルマントを頂点として線で結んだ「四辺形」**である。話者グループ（ここでは性別）ごとに別の四辺形を描き、輪郭の違いを比べる。生のHzと正規化後を並べて描き、前節で数値として確認した「正規化で話者差が消える」ことを図でも確認する。

In [ ]:
# 母音の並び順を、母音空間を一周する順に指定しておく
# （そうしないとgeom_path()が頂点をアルファベット順につないでしまい、
#   星形のような意味のない図になる）。
vowel_order <- c("i", "I", "E", "{", "A", "O", "U", "u")

make_vowel_means <- function(data, f1_col, f2_col) {
  data %>%
    filter(vowel %in% vowel_order, talker_type %in% c("男性", "女性")) %>%
    mutate(vowel = factor(vowel, levels = vowel_order)) %>%
    group_by(talker_type, vowel) %>%
    summarise(F1 = mean(.data[[f1_col]]), F2 = mean(.data[[f2_col]]), .groups = "drop") %>%
    arrange(talker_type, vowel)
}

means_raw  <- make_vowel_means(pb,      "f1",  "f2")  %>% mutate(panel = "生のHz")
means_norm <- make_vowel_means(pb_norm, "f1z", "f2z") %>% mutate(panel = "正規化後")
means_both <- bind_rows(means_raw, means_norm) %>%
  mutate(panel = factor(panel, levels = c("生のHz", "正規化後")))

ggplot(means_both, aes(x = F2, y = F1, color = talker_type, group = talker_type)) +
  geom_path(linewidth = 0.8) +               # 頂点を順につないで四辺形にする
  geom_point(size = 2.5) +
  geom_text(aes(label = vowel), vjust = -0.8, show.legend = FALSE) +
  facet_wrap(~panel, scales = "free") +      # 生Hzと正規化後を並べる。軸の単位が違うのでscales="free"
  scale_x_reverse() +
  scale_y_reverse() +
  theme_minimal(base_size = 14, base_family = "notosans") +
  labs(title = "母音四辺形：生のHz vs 正規化後（性別で比較）",
       x = "F2  ←前舌　　後舌→", y = "F1  ←狭い　　広い→", color = "話者タイプ")

# --- 実行結果 ---
# 左右2枚の母音四辺形が並んで描画される（生Hz／正規化後、それぞれ男性・女性の輪郭）。
# テキスト出力は無い。
#
# 【図の読み方】
# ・**geom_path()はデータフレームの行の並び順どおりに線をつなぐ**。
#   `arrange(talker_type, vowel)` と `factor(vowel, levels = vowel_order)` の
#   2つがそろって初めて、母音空間を一周する自然な順序で線がつながる。
#   どちらかを忘れると、四辺形ではなく線が交差した図になる。
# ・**左の「生のHz」パネルでは、女性の輪郭が男性の輪郭より右上（F1・F2とも
#   高い側）にずれて見える。** 前節で /i/ について確認した「生のHzでは884Hz
#   高い」という差（声道が短いことによる解剖学的な差）が、母音全体に
#   一貫して現れている。
# ・**右の「正規化後」パネルでは、男女の輪郭がほぼ重なる。** これも前節の
#   数値（正規化後のF2差は-0.023 SD、実質ゼロ）と対応する。
# ・輪郭の**形そのもの**（頂点の相対的な配置）はどちらのパネルでも男女で
#   似ている。声道の長さが違っても、母音どうしの相対的な区別のしかたは
#   共通しているということである。正規化は「形」はそのままに、
#   「話者ごとの絶対位置」だけを揃える操作だと、この対比から視覚的に
#   確認できる。


■ ヘルツのまま扱ってよいか：セミトーン・Bark変換：

音高・音色の知覚はヘルツに対して線形ではない。F0はセミトーン（対数）、フォルマントはBark/ERB/Melで、知覚的な等間隔性に近づけられる。

In [ ]:
# F0: 同じ「10%上昇」でも、低い声域と高い声域でHzの差は2倍違うが、
# セミトーン（12 * log2(比）で定義）では完全に一致する。
semitone <- function(f, ref = 100) 12 * log2(f / ref)
cat("低い声域: 100Hz→110Hz  Hz差", 110-100, " ST差", semitone(110)-semitone(100), "\n")
cat("高い声域: 200Hz→220Hz  Hz差", 220-200, " ST差", semitone(220)-semitone(200), "\n")

# フォルマント: Peterson & Barney (1952) の実測データでBark変換の圧縮を確認する。
library(phonTools)
hz_to_bark <- function(f) 26.81 / (1 + 1960/f) - 0.53
pb_means <- pb52 %>% group_by(vowel) %>% summarise(F1 = mean(f1)) %>%
  mutate(F1_bark = hz_to_bark(F1))
i_row <- pb_means %>% filter(vowel == "i"); I_row <- pb_means %>% filter(vowel == "I")
A_row <- pb_means %>% filter(vowel == "A"); V_row <- pb_means %>% filter(vowel == "V")
cat("\n/i/-/I/: Hz差", round(I_row$F1 - i_row$F1, 1),
    " Bark差", round(I_row$F1_bark - i_row$F1_bark, 2),
    " Bark/Hz", round((I_row$F1_bark - i_row$F1_bark) / (I_row$F1 - i_row$F1), 5), "\n")
cat("/V/-/A/: Hz差", round(A_row$F1 - V_row$F1, 1),
    " Bark差", round(A_row$F1_bark - V_row$F1_bark, 2),
    " Bark/Hz", round((A_row$F1_bark - V_row$F1_bark) / (A_row$F1 - V_row$F1), 5), "\n")

# --- 実行結果 ---
# 低い声域: 100Hz→110Hz  Hz差 10  ST差 1.650042
# 高い声域: 200Hz→220Hz  Hz差 20  ST差 1.650042
#
# /i/-/I/: Hz差 136.9  Bark差 1.33  Bark/Hz 0.00969
# /V/-/A/: Hz差 111.6  Bark差 0.78  Bark/Hz 0.00701
#
# 【出力の読み方】
# ・F0の2行は、Hzでは2倍違う差（10Hzと20Hz）が、セミトーンでは1.650042と
#   完全に一致することを示す。音高の知覚は比に対応するので、話者間で声域が
#   違っても、セミトーンなら「同じ大きさの変化」を同じ数値で表せる。
# ・フォルマントの2行は、同じ1Hzあたりの知覚的な変化量（Bark/Hz）が、
#   低いF1側（/i/-/I/、高母音）で0.00969、高いF1側（/V/-/A/、低母音）で
#   0.00701と、**低い周波数帯のほうが大きい**ことを示す。これがBark尺度が
#   「高域を圧縮する」と言われる性質の実体である。
# ・話者間で音高・音色の変化の大きさを比べる研究では、ヘルツではなく
#   これらの変換後の尺度を使うのが標準的である（詳細は本文2.6節）。


■ 持続時間は対数変換してから扱う：

VOTや音節長のような持続時間は、下限がゼロで右に裾を引く分布になりやすい。対数正規分布に近い音節長の擬似データで確かめる。

In [ ]:
install.packages("e1071", repos = "https://cloud.r-project.org")  # skewness()用
set.seed(21)
dur <- rlnorm(300, meanlog = log(180), sdlog = 0.35)  # 中央値180ms付近の音節長

cat("生の持続時間  : 歪度", round(e1071::skewness(dur), 3),
    " Shapiro-Wilk p =", format.pval(shapiro.test(dur)$p.value, digits = 3), "\n")
cat("log(持続時間) : 歪度", round(e1071::skewness(log(dur)), 3),
    " Shapiro-Wilk p =", format.pval(shapiro.test(log(dur))$p.value, digits = 3), "\n")

cat("\n算術平均:", round(mean(dur), 1), " 中央値:", round(median(dur), 1),
    " 幾何平均:", round(exp(mean(log(dur))), 1), "\n")

# --- 実行結果 ---
# 生の持続時間  : 歪度 0.755  Shapiro-Wilk p = 1.14e-06
# log(持続時間) : 歪度 -0.181  Shapiro-Wilk p = 0.491
#
# 算術平均: 190.6  中央値: 183.4  幾何平均: 180
#
# 【出力の読み方】
# ・生のままでは歪度0.755で、Shapiro-Wilk検定も正規性を明確に棄却する
#   （p = 1.14e-06）。対数変換すると歪度がほぼゼロになり、正規性を棄却する
#   証拠も無くなる（p = 0.491）。**対数変換した持続時間に線形モデルを当てるのが
#   既定の出発点になる**——第11章で反応時間に対数変換を使った理由と同じ構造。
# ・対数変換した値の平均を指数で戻すと幾何平均（180.0）になり、生の値の
#   算術平均（190.6）より中央値（183.4）に近い。歪んだ分布では、算術平均は
#   少数の長い値に引きずられる。
# ・発話速度が変わると、測っている現象と無関係にあらゆる持続時間が伸び縮みする。
#   対処は、発話速度を共変量としてモデルに含めるか、個々の持続時間を発話速度で
#   正規化してから分析するかのいずれかである（詳細は本文2.7節）。
# ・複数の発話にまたがる軌跡データを比較するときは、時間軸を0（開始）から
#   1（終了）に正規化してから測定する。第7章のGAMMのtime変数はこの
#   時間正規化を済ませたものである。


■ スペクトルモーメントと声質指標：

摩擦音や声質を扱う研究で中心になる指標。分布の性質がF0・フォルマントとは異なるので、線形モデルに投入する前に確認が要る。

In [ ]:
set.seed(30)
freq <- seq(0, 10000, by = 100)
power_s  <- dnorm(freq, mean = 6500, sd = 1200)
power_sh <- dnorm(freq, mean = 3500, sd = 1500)
power_s  <- power_s  + 0.05 * max(power_s)  * runif(length(freq))
power_sh <- power_sh + 0.05 * max(power_sh) * runif(length(freq))

spectral_moments <- function(freq, power) {
  p <- pmax(power, 0) / sum(pmax(power, 0))
  cog <- sum(freq * p)
  sd_ <- sqrt(sum((freq - cog)^2 * p))
  skew <- sum(((freq - cog)/sd_)^3 * p)
  kurt <- sum(((freq - cog)/sd_)^4 * p) - 3
  c(COG = cog, SD = sd_, skewness = skew, kurtosis = kurt)
}
spectral_moments(freq, power_s)   # /s/相当の擬似スペクトル
spectral_moments(freq, power_sh)  # /sh/相当の擬似スペクトル

# H1-H2: 差の指標は元の測定誤差が2乗和で拡大する
set.seed(31)
h1 <- rnorm(200, 30, 3); h2 <- rnorm(200, 25, 3)   # 測定誤差SD=3の擬似データ
cat("H1のSD:", round(sd(h1),2), " H2のSD:", round(sd(h2),2),
    " H1-H2のSD:", round(sd(h1-h2),2), "\n")

# --- 実行結果 ---
# /s/相当:  COG 6390.61  SD 1416.35  skewness -0.78  kurtosis  2.06
# /sh/相当: COG 3629.93  SD 1627.38  skewness  0.49  kurtosis  0.67
#
# H1のSD: 2.86  H2のSD: 3.11  H1-H2のSD: 4.17（理論値 sqrt(3^2+3^2)=4.24）
#
# 【出力の読み方】
# ・COG（重心）は生成時の平均（6500Hz, 3500Hz）に近い値を回収できている。
#   COGはヘルツと同じ比尺度の量であり、話者間比較には第2章で見た正規化の
#   問題がそのまま当てはまる。分布が右に歪みやすい点も持続時間と同様である。
# ・H1とH2それぞれの測定誤差SDが3程度でも、その差H1-H2のSDは4.17（理論値
#   √(3²+3²)≈4.24）まで拡大する。**差の指標は元の測定誤差が2乗和で乗る**
#   ため、H1やH2単体より測定誤差に敏感になる。
# ・HNRには理論的な上限があり、jitter・shimmerは負にならない比率に近い量である。
#   いずれも0や上限付近に値が集中しやすく、正規分布を前提とする線形モデルに
#   そのまま投入する前に分布の形を確認する。


■ 演習：

1. 話者ごとにVOTの分布が違うかを `facet_wrap(~speaker)` で確認せよ。
2. 歪度が大きい分布に出会ったとき、平均より中央値を報告すべき理由を説明せよ。
3. 棒グラフの例をgenderでも分け、`fill = condition`のまま`facet_wrap(~gender)`を追加して2要因を同時に見よ。

■ 演習の解答：

1と3は作図、2は考え方を問う問題である。2の答えは出力ではなくコメントに書いてある。

In [ ]:
# 1. 話者ごとに分布が違うかをfacet_wrap()で並べて確認する。
#    facet_wrap(~speaker)は「話者ごとに小さいパネルを1枚ずつ作る」という意味。
ggplot(vot_data, aes(x = VOT, fill = condition)) +
  geom_histogram(bins = 20, alpha = 0.6, position = "identity") +
  facet_wrap(~speaker) +
  theme_minimal(base_size = 14, base_family = "notosans")

# 2. 「歪度が大きいとき、平均より中央値を報告すべき理由」
#    → 平均は分布の裾に引きずられるが、中央値は引きずられないため。
#    実際に外れ値を1つ足して、2つの指標がどれだけ動くかを比べてみる。
vot_voiced <- vot_data$VOT[vot_data$condition == "voiced"]
vot_outlier <- c(vot_voiced, 300)   # 300msという極端な値を1つだけ足す

cat("外れ値なし: 平均", round(mean(vot_voiced), 1),
    "/ 中央値", round(median(vot_voiced), 1), "\n")
cat("外れ値あり: 平均", round(mean(vot_outlier), 1),
    "/ 中央値", round(median(vot_outlier), 1), "\n")
# 平均は大きく動くのに中央値はほとんど動かない。これが「歪んだ分布では
# 中央値を報告する」根拠である。測定ミス由来の外れ値が混ざりうる音響測定では
# とくに効いてくる。

# 3. genderとconditionを同時に見る。
#    fill=conditionで色を条件に、facet_wrap(~gender)でパネルを性別に割り当てる。
vot_data %>%
  group_by(gender, condition) %>%
  summarise(mean_VOT = mean(VOT), .groups = "drop") %>%
  ggplot(aes(x = condition, y = mean_VOT, fill = condition)) +
  geom_col() +
  facet_wrap(~gender) +
  theme_minimal(base_size = 14, base_family = "notosans")

# --- 実行結果（2問目のcat出力。1問目と3問目は図として描画される）---
# 外れ値なし: 平均 22.9 / 中央値 22.3
# 外れ値あり: 平均 25.6 / 中央値 22.4
#
# 【出力の読み方】
# ・300msという極端な値を**1つ**足しただけで、平均は22.9→25.6と2.7msも動いた。
#   一方、中央値は22.3→22.4とほとんど動いていない（0.1ms）。
#   これが2問目の答えである。平均は全部の値を足して割るので外れ値の影響を
#   まともに受けるが、中央値は「真ん中に来る値」なので、端に何が入っても
#   順位が1つずれるだけで済む。
# ・音響測定では、セグメンテーションのずれや無音区間の誤検出で桁違いの値が
#   紛れ込むことがある。**外れ値を消す前の記述統計では、平均と中央値を
#   並べて報告する**と、分布が歪んでいるかどうかが読者にも伝わる。
# ・1問目の図（話者ごとのヒストグラム）では、どの話者のパネルにも山が2つ
#   （voicedとvoicelessの塊）現れるが、山の位置と離れ方は話者ごとに違う。
# ・3問目の図には**罠がある**。voiceless条件の棒が、Fのパネルで約70ms、
#   Mのパネルで約81msと、10msほど違って見える。しかし本書のデータ生成コード
#   （第1章）にgenderの効果は一切入れていない。ではなぜ差が出るのか。
#   S01〜S05をM、S06〜S10をFに割り当てているため、**たまたまMに「有声と無声を
#   大きく作り分ける話者」が多く入った**だけである。つまりこれは性差ではなく
#   話者のサンプリングのばらつきが性別ごとに集計されて見えているにすぎない。
#   話者10人を性別で5人ずつに分けた集計は、実質「5対5の比較」でしかない。
#   **棒グラフは各話者が何人分の情報から来ているかを隠す**ので、こうした
#   見かけの差が生まれる。第5章で (1 | speaker) を入れると、この差が
#   話者間のばらつきとして正しく吸収され、性別の効果としては残らなくなる。
